In [ ]:
# Book Recommendation System using KNN

# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Step 2: Load the Dataset (already loaded in project environments)
# These variables should already be available:
# ratings, books, users

# Merge datasets (if not already merged)
book_ratings = pd.merge(ratings, books, on="ISBN")
book_ratings = pd.merge(book_ratings, users, on="User-ID")

# Step 3: Filter out less active users and unpopular books
active_users = book_ratings['User-ID'].value_counts() > 200
popular_books = book_ratings['Book-Title'].value_counts() > 100

book_ratings = book_ratings[book_ratings['User-ID'].isin(active_users[active_users].index)]
book_ratings = book_ratings[book_ratings['Book-Title'].isin(popular_books[popular_books].index)]

# Step 4: Create a pivot table (book-user matrix)
book_user_matrix = book_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
book_user_matrix = book_user_matrix.fillna(0)

# Step 5: Fit KNN model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_user_matrix)

# Step 6: Create get_recommends() function
def get_recommends(book_title):
    try:
        index = book_user_matrix.index.get_loc(book_title)
    except KeyError:
        return [book_title, [["Book not found in dataset", None]]]

    distances, indices = model.kneighbors(book_user_matrix.iloc[index, :].values.reshape(1, -1), n_neighbors=6)

    recommends = []
    for i in range(1, len(distances[0])):  # Skip the first, it's the same book
        recommends.append([book_user_matrix.index[indices[0][i]], distances[0][i]])

    return [book_title, recommends]

# Step 7: Test the function
get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")